<div dir="rtl" style="text-align:right; line-height:2; font-size:17px; max-width:900px;">

# 📦 ضغط ملفات تيليجرام

### الاستخدام اليومي
1. ابعت ملف الصوت أو الفيديو في قناة **📦 Smart Compressor** على تيليجرام.
2. ارجع هنا واضغط زر **▶ التشغيل**.
3. النتيجة هترجع لنفس القناة تلقائيًا.

### إعادة معالجة ملف قديم
اعمل **رد** على الملف داخل تيليجرام، واكتب واحد من الأوامر دي:

- **🔁** — إعادة المعالجة بنفس الوضع الحالي.
- **🔁 أصغر** — ضغط الفيديو بشكل أقوى.
- **🔁 صوت** — استخراج صوت صغير جدًا.
- **🔁 80** — استهداف حجم قريب من 80 ميجابايت.

> أول تشغيل فقط هيطلب بيانات تيليجرام، وبعدها الإعدادات تتحفظ في Google Drive الخاص بيك.

</div>


In [ ]:
#@title ▶ تشغيل ضاغط الملفات
#@markdown اختار الوضع فقط لو محتاج تغيّر الإعداد الافتراضي.
الوضع = "تلقائي — مناسب لمعظم الاستخدامات" #@param ["تلقائي — مناسب لمعظم الاستخدامات", "صوت صغير جدًا", "فيديو متوازن", "فيديو سريع", "أصغر حجم للفيديو", "حجم فيديو محدد"]
الحجم = "" #@param {type:"string"}

import os
import re
import subprocess
import urllib.request
import json
import time

REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "telegram-smart-compressor/engine.py"
ENGINE_PATH = "/content/telegram_smart_compressor_engine.py"

os.environ["TSC_PROFILE"] = str(الوضع or "تلقائي — مناسب لمعظم الاستخدامات")
os.environ["TSC_TARGET_SIZE_MB"] = str(الحجم or "").strip()
os.environ["TSC_UPDATE_CHANNEL"] = BRANCH

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha

    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "Smart-Compressor-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))["object"]["sha"]

def download_engine(sha):
    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Smart-Compressor-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode("utf-8")

try:
    print("🔄 جاري تحميل أحدث نسخة مستقرة...")
    sha = latest_sha()
    engine = download_engine(sha)
    m = re.search(r'ENGINE_BUNDLE_VERSION\s*=\s*"([^"]+)"', engine)

    if len(engine) < 5000 or not m:
        raise RuntimeError("invalid-engine")

    print(f"✅ النسخة جاهزة: {m.group(1)}")

    with open(ENGINE_PATH, "w", encoding="utf-8") as f:
        f.write(engine)

    exec(compile(engine, ENGINE_PATH, "exec"), globals(), globals())

except SyntaxError:
    print("❌ E002")
    print("النسخة المحمّلة فيها خطأ برمجي. ابعت كود الخطأ E002.")

except Exception:
    print("❌ E001")
    print("تعذر تحميل المحرك من GitHub. تأكد من اتصال الإنترنت وجرب مرة أخرى.")
